# 第二次作业 04 来源审计与日期行程搜索

审计原始响应，按日期和实际车站搜索候选行程，并检查运行与等待时间。

原编号与新文件名的对应关系、外部依赖和执行说明见 [第二次作业导航](hw02_README.md)。

先运行本文件的**环境初始化**单元格；重启内核后也应先运行初始化。各节按原顺序排列，依赖上一步生成的变量时会在节前列出。

已有输出和执行编号是整理前的运行记录，未重新生成；新增初始化单元格没有历史输出。

[上一份 03](hw02_03_collection_resume.ipynb) · [下一份 05](hw02_05_followup_collection.ipynb)

## 环境初始化

此单元格集中加载公共依赖并检查脚本位置；不执行采集、安装或模型拟合。

In [ ]:
# 标准库
import importlib
import sys
from pathlib import Path

# 第三方库
import pandas as pd
from IPython.display import display

# 仅在这里配置本机项目位置。脚本和来源数据仍保留在原 railway_analysis 项目中。
ROOT = Path(r"D:\yangz\大三上\机器学习\第二次作业\railway_analysis")
EXPECTED_PYTHON = Path(r"D:\anaconda\envs\pda312\python.exe")
SCRIPTS_DIR = ROOT / "scripts"

if Path(sys.executable).resolve() != EXPECTED_PYTHON.resolve():
    raise RuntimeError("请选择 pda312 内核，再从本文件的环境初始化单元格开始运行。")
if not SCRIPTS_DIR.is_dir():
    raise FileNotFoundError(f"未找到作业脚本目录：{SCRIPTS_DIR}。请核对上方 ROOT 配置。")

REQUIRED_LOCAL_MODULES = (
    "stage1_collect",
    "stage1c_parser_v2",
    "stage1d_audit",
    "stage1d_routes",
)
missing_modules = [
    name for name in REQUIRED_LOCAL_MODULES
    if not (SCRIPTS_DIR / f"{name}.py").is_file()
]
if missing_modules:
    raise FileNotFoundError(f"作业脚本缺失：{', '.join(missing_modules)}")

# 重复运行初始化时不重复插入同一路径；业务模块仍在对应步骤导入和 reload。
scripts_path = str(SCRIPTS_DIR)
if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)


## 辅助准备：加载路径、版本与本批模块【无需单独截图】

In [1]:
import stage1_collect
import stage1c_parser_v2
import stage1d_audit
import stage1d_routes
for module in (stage1_collect, stage1c_parser_v2, stage1d_audit, stage1d_routes):
    importlib.reload(module)
print("解释器：", sys.executable)
print("本批离线分析数据：2026-11-18，user_stage1c_v2_20261113_132207_325427")


解释器： D:\anaconda\envs\pda312\python.exe
本批离线分析数据：2026-11-18，user_stage1c_v2_20261113_132207_325427


## 问题1—核心代码10：全部995组原始数据核验

前置变量或模块：`stage1d_audit`。

In [2]:
# 功能：验证原始文件哈希、逐条重放解析、对照22137条CSV和34节点诊断表。
# 说明：内部一致性检查通过，不代表全铁路时刻表完整或候选站点已全部审定。
audit_result = stage1d_audit.audit(actor="user")
AUDIT_OUT = Path(audit_result["output_dir"])
for key in ["verified_queries", "train_records", "unknown_timetable_rows", "unresolved_station_rows",
            "raw_and_dictionary_files_hash_verified", "observed_core_stations", "observed_stations_outside_draft", "passed"]:
    print(f"{key}: {audit_result[key]}")
display(pd.Series(audit_result["query_status_counts"], name="count").rename_axis("query_status").to_frame())
print("核验输出目录：", AUDIT_OUT)


verified_queries: 995
train_records: 22137
unknown_timetable_rows: 88
unresolved_station_rows: 10
raw_and_dictionary_files_hash_verified: 998
observed_core_stations: 136
observed_stations_outside_draft: 55
passed: True


,count
query_status,
validated_records,793
partial_timetable_unavailable,63
no_direct_train_returned_not_unreachable,134
partial_station_metadata_unresolved,5


核验输出目录： D:\yangz\大三上\机器学习\第二次作业\railway_analysis\data\processed\user_stage1d_audit_20261113_141603_610163


## 问题1—核心代码11：有时间约束的铁路行程搜索

前置变量或模块：`AUDIT_OUT`、`stage1d_routes`。

In [3]:
# 功能：按实际站码和完整日期构造时间事件图，搜索已采集网络内的最小历时。
# 注意：全部当前上车记录都是11月18日，缺少后续日期发车信息。
#       有路径只是已采集图内的候选，未找到不等于物理不可达；不能声称全国全时刻表最短。
routing = stage1d_routes.run(actor="user", audit_dir=AUDIT_OUT)
ROUTE_OUT = Path(routing["output_dir"])
comparison = pd.read_csv(ROUTE_OUT / "routing_quality_comparison.csv", encoding="utf-8-sig")
display(comparison[["scenario", "finite_directed_city_pairs", "missing_directed_city_pairs",
                    "pairs_using_transfer", "seconds"]])
print("行程与来源目录：", ROUTE_OUT)
print("matrix_ready:", routing["matrix_ready"], "（还需补充日期与特殊地区资料）")


时间图分析： draft83_60
已采集时刻表中找到的城市对： 933 /992
时间图分析： observed_group_30
已采集时刻表中找到的城市对： 936 /992
时间图分析： observed_group_60
已采集时刻表中找到的城市对： 934 /992
时间图分析： observed_group_90
已采集时刻表中找到的城市对： 932 /992


,scenario,finite_directed_city_pairs,missing_directed_city_pairs,pairs_using_transfer,seconds
0,draft83_60,933,59,361,2.027901
1,observed_group_30,936,56,439,2.260476
2,observed_group_60,934,58,361,2.256578
3,observed_group_90,932,60,298,2.054972


行程与来源目录： D:\yangz\大三上\机器学习\第二次作业\railway_analysis\data\processed\user_stage1d_routes_20261113_141703_918913
matrix_ready: False （还需补充日期与特殊地区资料）


## 问题1—核心代码12：逐段行程证据与定向补查清单

前置变量或模块：`ROUTE_OUT`。

In [4]:
# 功能：展示一条真实候选，复核“全程＝运行＋等待”，并列出仍需补查的日期和方向。
# 计划只优先针对当前未找到的城市对，查询可到达中转城市的最早后续日期。
# 它不是全部必要补采的穷尽清单，也没有在本段真的联网；已找到候选以后仍可能被更快路线改善。
routes = pd.read_csv(ROUTE_OUT / "route_scenarios.csv", encoding="utf-8-sig")
legs = pd.read_csv(ROUTE_OUT / "route_legs_with_sources.csv", encoding="utf-8-sig")
finite = routes.loc[(routes["scenario"] == "observed_group_60") & (routes["transfers"] > 0)]
if not finite.empty:
    example = finite.sort_values(["model_time_min", "from_node", "to_node"]).head(1)
    row = example.iloc[0]
    selected = legs.loc[(legs["scenario"] == row["scenario"]) &
                        (legs["from_node"] == row["from_node"]) & (legs["to_node"] == row["to_node"])].sort_values("leg_index")
    assert int(selected["duration_min"].sum() + selected["wait_before_min"].sum()) == int(row["model_time_min"])
    display(example[[
        "from_city", "to_city", "model_time_min", "running_min", "wait_min", "transfers",
    ]])
    display(selected[[
        "leg_index", "train_code", "from_name", "to_name", "departure_at", "arrival_at",
        "wait_before_min",
    ]])
    example.to_csv(ROUTE_OUT / "representative_itinerary.csv", index=False, encoding="utf-8-sig")
    selected.to_csv(ROUTE_OUT / "representative_itinerary_legs.csv", index=False, encoding="utf-8-sig")
plan = pd.read_csv(ROUTE_OUT / "next_date_query_plan.csv", encoding="utf-8-sig")
print("下一轮定向补查组数：", len(plan), "；本段不发起查询。")
display(plan[["query_date", "from_city", "to_city", "affected_od_count"]].head(10))


,from_city,to_city,model_time_min,running_min,wait_min,transfers
2295,杭州,天津,303.0,241.0,62.0,1.0


,leg_index,train_code,from_name,to_name,departure_at,arrival_at,wait_before_min
5333,1,G42,杭州东,南京南,2026-11-18T13:56:00+08:00,2026-11-18T14:59:00+08:00,0
5334,2,G22,南京南,天津南,2026-11-18T16:01:00+08:00,2026-11-18T18:59:00+08:00,62


下一轮定向补查组数： 436 ；本段不发起查询。


,query_date,from_city,to_city,affected_od_count
0,2026-11-19,南昌,拉萨,3
1,2026-11-19,广州,拉萨,3
2,2026-11-19,武汉,拉萨,3
3,2026-11-19,郑州,拉萨,3
4,2026-11-19,郑州,福州,3
5,2026-11-19,重庆,拉萨,3
6,2026-11-19,重庆,福州,3
7,2026-11-19,长沙,拉萨,3
8,2026-11-19,上海,拉萨,2
9,2026-11-19,兰州,拉萨,2
